# Well Played, Mauer: Situational RBI

Do players have control over whether they come to the plate with runners on base? The answer seems obvious, but let's try to show strong evidence for such.

In [1]:
# import the necessary packages
import os
import sys
import pandas as pd

In [2]:
# set up the file paths
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
raw_data_dir = os.path.join(project_root, 'data', 'raw')
processed_data_dir = os.path.join(project_root, 'data', 'processed')

# test the paths
# print(f'Project Root: {project_root}')
# print(f'Raw Data Directory: {raw_data_dir}')
# print(f'Processed Data Directory: {processed_data_dir}')

In [3]:
# read in the csv for all qualified seasons from 2006 - 2015
# data courtesy of stathead
filename = 'mlb_men_on_split_2006_2015.csv'
csv_path = os.path.join(raw_data_dir, filename)
batters = pd.read_csv(csv_path)
batters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1508 entries, 0 to 1507
Data columns (total 36 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rk                 1508 non-null   int64  
 1   I                  0 non-null      float64
 2   Player             1508 non-null   object 
 3   Split              1508 non-null   object 
 4   Year               1508 non-null   int64  
 5   G                  1508 non-null   int64  
 6   PA                 1508 non-null   int64  
 7   PAtot              1508 non-null   int64  
 8   %                  1508 non-null   float64
 9   GS                 0 non-null      float64
 10  AB                 1508 non-null   int64  
 11  R                  1508 non-null   int64  
 12  H                  1508 non-null   int64  
 13  2B                 1508 non-null   int64  
 14  3B                 1508 non-null   int64  
 15  HR                 1508 non-null   int64  
 16  RBI                1508 

In [4]:
# sort by player and season
batters = batters.sort_values(by=['Player-additional', 'Year'])

In [5]:
# create a shifted dataframe
shifted = batters[['Player', 'Player-additional', 'Year', 'PA', '%', 'H', 'RBI', 'BA', 'OBP', 'SLG']].copy()
shifted['Year'] = shifted['Year'] - 1  # shift backward to match with previous season
shifted = shifted.rename(columns={
    'PA':'PA_next',
    '%':'%_next',
    'H':'H_next',
    'RBI':'RBI_next',
    'BA':'BA_next',
    'OBP':'OBP_next',
    'SLG':'SLG_next'
})
shifted.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1508 entries, 166 to 1167
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Player             1508 non-null   object 
 1   Player-additional  1508 non-null   object 
 2   Year               1508 non-null   int64  
 3   PA_next            1508 non-null   int64  
 4   %_next             1508 non-null   float64
 5   H_next             1508 non-null   int64  
 6   RBI_next           1508 non-null   int64  
 7   BA_next            1508 non-null   float64
 8   OBP_next           1508 non-null   float64
 9   SLG_next           1508 non-null   float64
dtypes: float64(4), int64(4), object(2)
memory usage: 129.6+ KB


In [6]:
merged = pd.merge(batters, shifted, on=['Player-additional', 'Year'], how='inner')
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 44 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rk                 882 non-null    int64  
 1   I                  0 non-null      float64
 2   Player_x           882 non-null    object 
 3   Split              882 non-null    object 
 4   Year               882 non-null    int64  
 5   G                  882 non-null    int64  
 6   PA                 882 non-null    int64  
 7   PAtot              882 non-null    int64  
 8   %                  882 non-null    float64
 9   GS                 0 non-null      float64
 10  AB                 882 non-null    int64  
 11  R                  882 non-null    int64  
 12  H                  882 non-null    int64  
 13  2B                 882 non-null    int64  
 14  3B                 882 non-null    int64  
 15  HR                 882 non-null    int64  
 16  RBI                882 non

In [7]:
merged = merged.drop(columns=['Player_y']).rename(columns={'Player_x': 'Player'})
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 43 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rk                 882 non-null    int64  
 1   I                  0 non-null      float64
 2   Player             882 non-null    object 
 3   Split              882 non-null    object 
 4   Year               882 non-null    int64  
 5   G                  882 non-null    int64  
 6   PA                 882 non-null    int64  
 7   PAtot              882 non-null    int64  
 8   %                  882 non-null    float64
 9   GS                 0 non-null      float64
 10  AB                 882 non-null    int64  
 11  R                  882 non-null    int64  
 12  H                  882 non-null    int64  
 13  2B                 882 non-null    int64  
 14  3B                 882 non-null    int64  
 15  HR                 882 non-null    int64  
 16  RBI                882 non

In [8]:
# check the correlation coefficients
corr = merged['%'].corr(merged['%_next'])
print(f'% of PA with Men On r = {corr:.3f}')

corr = merged['PA'].corr(merged['PA_next'])
print(f'PA with Men On r = {corr:.3f}')

corr = merged['H'].corr(merged['H_next'])
print(f'H with Men On r = {corr:.3f}')

corr = merged['RBI'].corr(merged['RBI_next'])
print(f'RBI r = {corr:.3f}')

corr = merged['BA'].corr(merged['BA_next'])
print(f'AVG r = {corr:.3f}')

corr = merged['OBP'].corr(merged['OBP_next'])
print(f'OBP r = {corr:.3f}')

corr = merged['SLG'].corr(merged['SLG_next'])
print(f'SLG r = {corr:.3f}')

% of PA with Men On r = 0.659
PA with Men On r = 0.503
H with Men On r = 0.444
RBI r = 0.573
AVG r = 0.302
OBP r = 0.456
SLG r = 0.453


In [9]:
corr = merged['PA'].corr(merged['RBI'])
print(f'r = {corr:.3f}')

r = 0.791


In [17]:
# create the path to save the csv
processed_path = os.path.join(processed_data_dir, 'mlb_men_on_split_2006_2015_processed.csv')

# save to csv
merged.to_csv(processed_path, index=False)